# Fashion-Rec: Compute E5 Embeddings on Colab GPU

This notebook computes embeddings for products exported by
`scripts/export_embedding_batch.py`, using the **exact same** model,
prefixes, and normalization as the production `HuggingFaceE5Provider`
(`app/providers/e5.py`). It does not touch the local PostgreSQL database -
it only reads an uploaded input Parquet file and produces an output
Parquet file for you to download and import locally with
`scripts/import_embeddings.py`.

**This notebook is isolated from the production application**: it does not
import from or modify any file in the real repo. The provider code below is
a manual, byte-identical copy (see the cell's header comment) - it is not
imported from the repo and does not require git/repo access from Colab.

**What you need to do:**
1. Runtime > Change runtime type > GPU.
2. Run all cells in order.
3. When prompted, upload the Parquet file produced by
   `scripts/export_embedding_batch.py`.
4. At the end, download the two output files (`embedding_result.parquet`,
   `manifest.json`) and move them back to your local machine.
5. Run `scripts/import_embeddings.py --input embedding_result.parquet`
   locally.

No Google Drive mounting, no database credentials, and no network access to
your local Postgres are used anywhere in this notebook.


## 1. Install dependencies

Colab ships torch preinstalled; only sentence-transformers needs installing.

In [ ]:
!pip install -q "sentence-transformers==3.0.1"


## 2. Upload the input Parquet

Produced locally by `scripts/export_embedding_batch.py --output embedding_export.parquet`.

In [ ]:
from google.colab import files

uploaded = files.upload()
input_filename = next(iter(uploaded.keys()))
print(f"Uploaded: {input_filename}")


## 3. Embedding provider (manual copy of app/providers/base.py + app/providers/e5.py)

In [ ]:
# =============================================================================
# MANUAL COPY of app/providers/base.py and app/providers/e5.py
#
# These two classes are byte-identical copies of the production provider
# code, included here so this notebook runs with ZERO dependency on the
# local repo/DB (no git clone, no PYTHONPATH tricks needed in Colab).
#
# IMPORTANT: if app/providers/base.py or app/providers/e5.py ever change in
# the real codebase, this cell must be manually re-copied to match. Nothing
# enforces that automatically - check the source files before relying on
# this notebook for a new embedding run.
#
# Source of truth:
#   app/providers/base.py
#   app/providers/e5.py
# =============================================================================
import logging
from abc import ABC, abstractmethod
from typing import Any

logger = logging.getLogger(__name__)


class EmbeddingProvider(ABC):
    """Abstract base class for embedding providers."""

    @property
    @abstractmethod
    def dim(self) -> int:
        """Embedding dimension."""

    @abstractmethod
    def embed_passages(self, texts: list[str]) -> list[list[float]]:
        """Embed passages for indexing.

        Args:
            texts: List of text passages to embed.

        Returns:
            List of embeddings, each a list of floats.
        """

    @abstractmethod
    def embed_queries(self, texts: list[str]) -> list[list[float]]:
        """Embed queries for search.

        Args:
            texts: List of search queries to embed.

        Returns:
            List of embeddings, each a list of floats.
        """


class HuggingFaceE5Provider(EmbeddingProvider):
    """Embedding provider using intfloat/multilingual-e5-base model.

    Features:
    - 768-dimensional embeddings
    - Multilingual (Hindi, English, etc.)
    - L2-normalized vectors
    - Query/passage prefix support (e5 requirement)
    - GPU acceleration with CPU fallback
    - Lazy model loading
    """

    def __init__(self, model_name: str = "intfloat/multilingual-e5-base", batch_size: int = 64):
        """Initialize provider.

        Args:
            model_name: HuggingFace model identifier.
            batch_size: Batch size for embedding.
        """
        self.model_name = model_name
        self.batch_size = batch_size
        self._model: Any = None
        self._device = self._get_device()
        logger.info(f"E5Provider initialized: model={model_name}, device={self._device}")

    def _get_device(self) -> str:
        """Detect device (cuda or cpu) with lazy torch import."""
        try:
            import torch
            return "cuda" if torch.cuda.is_available() else "cpu"
        except ImportError:
            return "cpu"

    def _load_model(self) -> None:
        """Lazy-load the embedding model."""
        if self._model is None:
            try:
                from sentence_transformers import SentenceTransformer
                logger.info(f"Loading model {self.model_name} on {self._device}...")
                self._model = SentenceTransformer(self.model_name, device=self._device)
                logger.info(f"Model loaded. Output dimension: {self._model.get_sentence_embedding_dimension()}")
            except ImportError as e:
                raise RuntimeError(f"sentence-transformers not installed: {e}") from e

    @property
    def dim(self) -> int:
        return 768

    def embed_passages(self, texts: list[str]) -> list[list[float]]:
        """Embed passages with 'passage: ' prefix."""
        return self._embed_texts([f"passage: {text}" for text in texts])

    def embed_queries(self, texts: list[str]) -> list[list[float]]:
        """Embed queries with 'query: ' prefix."""
        return self._embed_texts([f"query: {text}" for text in texts])

    def _embed_texts(self, texts: list[str]) -> list[list[float]]:
        """Embed texts in batches.

        Args:
            texts: List of (already-prefixed) texts to embed.

        Returns:
            List of normalized embeddings.
        """
        self._load_model()

        embeddings = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i : i + self.batch_size]
            try:
                batch_embeddings = self._model.encode(
                    batch,
                    convert_to_numpy=True,
                    normalize_embeddings=True,
                    show_progress_bar=False,
                )
                embeddings.extend(batch_embeddings.tolist())
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    # Handle OOM: clear cache and retry with smaller batch
                    logger.warning(f"OOM on batch size {len(batch)}, clearing cache and retrying")
                    try:
                        import torch
                        torch.cuda.empty_cache()
                    except ImportError:
                        pass

                    # Recursively retry with smaller batches
                    if len(batch) == 1:
                        raise RuntimeError(f"Cannot embed single text (OOM): {batch[0][:100]}") from e

                    smaller_batch_size = max(1, len(batch) // 2)
                    batch_embeddings = self._embed_texts_batch(batch, smaller_batch_size)
                    embeddings.extend(batch_embeddings)
                else:
                    raise

        return embeddings

    def _embed_texts_batch(self, texts: list[str], batch_size: int) -> list[list[float]]:
        """Embed texts with a specific batch size (used in OOM retry)."""
        embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            batch_embeddings = self._model.encode(
                batch,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
            embeddings.extend(batch_embeddings.tolist())
        return embeddings


## 4. Load the input Parquet and validate its schema

In [ ]:
import pandas as pd

df = pd.read_parquet(input_filename)

required_columns = {"id", "external_product_id", "content_hash", "search_text"}
missing = required_columns - set(df.columns)
assert not missing, f"Input Parquet is missing required columns: {missing}"

print(f"Loaded {len(df)} rows")
df.head()


## 5. Generate embeddings

`batch_size` only affects GPU throughput, not the resulting vectors -
HuggingFaceE5Provider's own batching/OOM-halving logic (copied above)
handles the actual encode calls, identical to the local provider. Colab
GPUs typically have more VRAM than a local GTX 1650, so a larger batch
size than the local default (64) is usually safe and faster - tune if you
hit an OOM.

In [ ]:
COLAB_BATCH_SIZE = 128  # local default is 64; Colab GPUs can usually go higher

provider = HuggingFaceE5Provider(
    model_name="intfloat/multilingual-e5-base",
    batch_size=COLAB_BATCH_SIZE,
)

search_texts = df["search_text"].tolist()
embeddings = provider.embed_passages(search_texts)

print(f"Generated {len(embeddings)} embeddings")


## 6. Sanity-check the output before writing it

In [ ]:
assert len(embeddings) == len(df), "Row count mismatch between input and embeddings"
assert all(len(e) == 768 for e in embeddings), "Found an embedding with dim != 768"

import math

assert all(
    all(math.isfinite(v) for v in e) for e in embeddings
), "Found a non-finite value in an embedding"

print("All embeddings are 768-dim and finite.")


## 7. Write the output Parquet + manifest

Output schema:

| column              | type                | notes                                |
|---------------------|---------------------|---------------------------------------|
| id                  | string              | pass-through, identity cross-check    |
| external_product_id | string              | pass-through, import match key        |
| content_hash        | string              | pass-through, unchanged               |
| embedding           | list<float32>[768]  | L2-normalized                         |


In [ ]:
import json
from datetime import datetime, timezone

output_df = pd.DataFrame({
    "id": df["id"],
    "external_product_id": df["external_product_id"],
    "content_hash": df["content_hash"],
    "embedding": embeddings,
})

output_path = "embedding_result.parquet"
output_df.to_parquet(output_path, index=False)

manifest = {
    "model": "intfloat/multilingual-e5-base",
    "dim": 768,
    "prefix": "passage: ",
    "normalized": True,
    "batch_size": COLAB_BATCH_SIZE,
    "row_count": len(output_df),
    "source_export_file": input_filename,
    "generated_at": datetime.now(timezone.utc).isoformat(),
}
with open("manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Wrote {output_path} ({len(output_df)} rows) and manifest.json")


## 8. Download the results

Move both files back to the repo locally, then run `scripts/import_embeddings.py --input embedding_result.parquet`.

In [ ]:
files.download("embedding_result.parquet")
files.download("manifest.json")
